In [ ]:
pip install rapidfuzz

In [33]:
import pandas as pd
from rapidfuzz import fuzz

def fuzzy_match_and_update_debit(df1, df2, threshold=80):
    # Ensure 'maincode' is index
    df1 = df1.set_index('code') if 'code' in df1.columns else df1
    df2 = df2.set_index('code') if 'code' in df2.columns else df2
    # Align on common maincodes
    common_maincodes = df1.index.intersection(df2.index)
    print(common_maincodes)
    # Align names
    names1 = df1.loc[common_maincodes, 'name'].astype(str).str.strip().str.upper().str.replace('&','AND').str.replace(r'[^A-Z0-9 ]', '', regex=True)
    print(names1)
    names2 = df2.loc[common_maincodes, 'name'].astype(str).str.strip().str.upper().str.replace('&','AND').str.replace(r'[^A-Z0-9 ]', '', regex=True)
    print(names2)

    # Fuzzy match each aligned row
    similarities = names1.combine(names2, lambda a, b: fuzz.ratio(a, b))

    # Convert to Series for easy filtering
    similarities = pd.Series(similarities, index=common_maincodes)

    # Find mismatches
    below_threshold = similarities[similarities < threshold].index

    # Set 'debit' = 123 in df2 for mismatches
    df2.loc[below_threshold, 'debit'] = 123

    return df2.reset_index()


In [34]:
df1 = pd.DataFrame({
    'code': [101, 102, 103],
    'name': ['John Smith', 'Jane Doe', 'Mike Jor co-operative AND LTD.']
})

df2 = pd.DataFrame({
    'code': [102, 101, 103],
    'name': ['Jane D.', ' Smi-th', 'Mike Jor co-operative & LTD.'],
    'debit': [1000, 2000, 3000]
})

updated_df2 = fuzzy_match_and_update_debit(df1, df2)
print(updated_df2)


Index([101, 102, 103], dtype='int64', name='code')
code
101                      JOHN SMITH
102                        JANE DOE
103    MIKE JOR COOPERATIVE AND LTD
Name: name, dtype: object
code
101                           SMITH
102                          JANE D
103    MIKE JOR COOPERATIVE AND LTD
Name: name, dtype: object
   code                          name  debit
0   102                       Jane D.   1000
1   101                        Smi-th    123
2   103  Mike Jor co-operative & LTD.   3000


In [16]:
from rapidfuzz import fuzz

print(fuzz.ratio("KANTIPURI SAVING & CREDIT CO-OPERATIVE LTD.", "KANTIPURI SAVING  CREDIT COOPERATIVE LTD"))  # Output: less than 100
print(fuzz.ratio("BHUSHAN AUTO PARTS & WORKSHOP.", "BHUSHAN AUTO PARTS AND WORKSHOP"))  # Output: less than 100


96.3855421686747
91.80327868852459
